# 05 — JEPA-Inspired Joint Embedding Model

**Purpose**: Train a multimodal longitudinal model that reasons about patient trajectories in embedding space.

All logic lives in `src/joint_embedding_model.py`. This notebook calls those functions and inspects the outputs.

**Architecture summary**:
- Three modality encoders: cognitive (MMSE, CDR), brain structure (nWBV, eTIV, ASF), demographic (Age, Education, SES, Sex)
- Visit 1 and visit 2 encoded separately with **shared encoder weights**
- Joint embeddings concatenated and passed to a predictor
- Prediction happens in embedding space — the model reasons about the *relationship between states*, not raw values

**Key JEPA principle applied**: learn representations of states, then predict across states.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from src.joint_embedding_model import (
    prepare_jepa_features,
    split_train_validation,
    train_jepa_model,
    predict_test_participants,
    save_jepa_predictions,
    DEVICE,
)
from src.ml_baseline import evaluate_predictions, summarise_metrics

RESULTS_DIR = project_root / "results"
print(f"Project root : {project_root}")
print(f"Device       : {DEVICE}")

## 1. Load train and test sets

In [ ]:
train_df = pd.read_csv(project_root / "data" / "processed" / "train_participants.csv")
test_df  = pd.read_csv(project_root / "data" / "processed" / "test_participants.csv")
print(f"Train: {len(train_df)} participants")
print(f"Test : {len(test_df)} participants")

## 2. Prepare features

Encode sex as integer and normalise continuous features. Fit on train only.

In [ ]:
train_prepared, test_prepared = prepare_jepa_features(train_df, test_df)
train_split_df, val_split_df  = split_train_validation(train_prepared)

print(f"Train split : {len(train_split_df)} participants")
print(f"Val split   : {len(val_split_df)} participants")
print(f"Test        : {len(test_prepared)} participants")

## 3. Train the model

In [ ]:
jepa_model = train_jepa_model(train_split_df, val_split_df)

## 4. Evaluate on test set

In [ ]:
y_pred, y_pred_proba = predict_test_participants(jepa_model, test_prepared)

metrics = evaluate_predictions(
    y_true=test_df["cdr_worsened_after_v2"],
    y_pred=y_pred,
    y_pred_proba=y_pred_proba,
)
summarise_metrics(metrics, model_name="JEPA-inspired Joint Embedding")

In [ ]:
# Prediction vs actual per participant
results_df = test_df[["subject_id", "participant_group", "cdr_worsened_after_v2"]].copy()
results_df["jepa_prediction"]       = y_pred
results_df["jepa_prediction_proba"] = y_pred_proba.round(3)
results_df

## 5. Save results

In [ ]:
save_jepa_predictions(test_df, y_pred, y_pred_proba, metrics, RESULTS_DIR)